# Sprint 3 — Inference: OULAD GRU Sequence Model

Same save-as-`.txt` → reload-from-`.txt` → infer pattern as the two XGBoost
inference notebooks — but the honesty check here is different and more
important to get right.

## Why `.txt` means something different for a PyTorch model

XGBoost's `.txt` "model" is at least genuine model data (UBJSON) that just
happens to have a misleading extension. A PyTorch model's saved state is a
**pickled dictionary of tensors** — `torch.save()` doesn't even look at the
file extension, it writes the identical binary content no matter what you
name the file. So `risk_model_oulad_gru_v1.txt` will be:

- **Just as functional** as the original `.pt` file — `torch.load()` doesn't
  care about extensions either, so reloading works identically.
- **Not remotely human-readable** — unlike XGBoost, there's no `dump_model()`
  equivalent that produces a meaningful text view of a GRU's weights; a raw
  float tensor dump wouldn't tell a human anything useful the way a decision
  tree's split conditions do.

So Section 6's "bonus readable version" here is different in kind from the
XGBoost notebooks' tree dump — it's a **human-readable architecture/config
summary**, not a dump of the actual weights, since the weights themselves
have no meaningful text representation.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT_ROOT = r"C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli"
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "oulad_data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")



RISK_THRESHOLD = 0.5  # model_gru_oulad.ipynb did not run a threshold-tuning pass;
                       # 0.5 is the default used there. Re-run that notebook's
                       # Section 6/7b-equivalent if you want a tuned cutoff for the GRU.

## 1. Model class definition (required to reload the state dict)

In [2]:
class EngagementGRU(nn.Module):
    """Identical to model_gru_oulad.ipynb's definition -- this class must exist
    wherever the checkpoint is loaded; torch.save() only stores the weights
    (state_dict), never the architecture code itself."""
    def __init__(self, input_size=2, hidden_size=32, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out, h_n = self.gru(x)
        final_h = h_n[-1]
        prob = self.head(final_h)
        return prob.squeeze(-1)

## 2. Load the trained checkpoint and save it as `.txt`

In [3]:
pt_path = os.path.join(MODELS_DIR, "risk_model_oulad_gru_v1.pt")
checkpoint = torch.load(pt_path, weights_only=False)
print(f"Loaded from: {pt_path}")
print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"Architecture config: input_size={checkpoint['input_size']}, "
      f"hidden_size={checkpoint['hidden_size']}, num_layers={checkpoint['num_layers']}")

txt_path = os.path.join(MODELS_DIR, "risk_model_oulad_gru_v1.txt")
torch.save(checkpoint, txt_path)
print(f"\nSaved as: {txt_path}")
print(f"File size: {os.path.getsize(txt_path):,} bytes "
      f"(compare to the original .pt -- should be identical, since torch.save() "
      f"ignores the extension entirely)")
print(f"Original .pt size: {os.path.getsize(pt_path):,} bytes")

Loaded from: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_oulad_gru_v1.pt
Checkpoint keys: ['model_state_dict', 'input_size', 'hidden_size', 'num_layers', 'max_weeks', 'feature_cols', 'log1p_clicks']
Architecture config: input_size=2, hidden_size=32, num_layers=1

Saved as: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_oulad_gru_v1.txt
File size: 20,077 bytes (compare to the original .pt -- should be identical, since torch.save() ignores the extension entirely)
Original .pt size: 20,077 bytes


## 3. Reload a fresh model *only* from the `.txt` file

In [4]:
checkpoint_from_txt = torch.load(txt_path, weights_only=False)

loaded_model = EngagementGRU(
    input_size=checkpoint_from_txt["input_size"],
    hidden_size=checkpoint_from_txt["hidden_size"],
    num_layers=checkpoint_from_txt["num_layers"],
)
loaded_model.load_state_dict(checkpoint_from_txt["model_state_dict"])
loaded_model.eval()
print("Model reloaded from .txt file and set to eval mode.")

MAX_WEEKS = checkpoint_from_txt["max_weeks"]
FEATURE_COLS = checkpoint_from_txt["feature_cols"]
LOG1P_CLICKS = checkpoint_from_txt["log1p_clicks"]
print(f"MAX_WEEKS={MAX_WEEKS}, FEATURE_COLS={FEATURE_COLS}, LOG1P_CLICKS={LOG1P_CLICKS}")

Model reloaded from .txt file and set to eval mode.
MAX_WEEKS=40, FEATURE_COLS=['total_clicks', 'days_active'], LOG1P_CLICKS=True


In [5]:
# Sanity check: identical random input through both the original state_dict
# and the .txt-reloaded one should give identical output
torch.manual_seed(0)
dummy_input = torch.randn(5, MAX_WEEKS, checkpoint["input_size"])

original_model = EngagementGRU(checkpoint["input_size"], checkpoint["hidden_size"], checkpoint["num_layers"])
original_model.load_state_dict(checkpoint["model_state_dict"])
original_model.eval()

with torch.no_grad():
    original_out = original_model(dummy_input)
    reloaded_out = loaded_model(dummy_input)

max_diff = (original_out - reloaded_out).abs().max().item()
print(f"Max output difference (.pt-loaded vs .txt-reloaded): {max_diff:.10f}")
assert max_diff < 1e-6, "Reloaded model does not match the original"
print("Confirmed: identical outputs. The .txt file round-trips perfectly.")

Max output difference (.pt-loaded vs .txt-reloaded): 0.0000000000
Confirmed: identical outputs. The .txt file round-trips perfectly.


## 4. Batch inference — score a real sample of students

Streams `studentVle.csv` filtered to a small sample of real students (spanning
all four `final_result` outcomes) rather than the full 26K population, since
this notebook is demonstrating the inference *call*, not re-running the full
training-scale pipeline. See `model_gru_oulad.ipynb` for the full-population
version.

In [6]:
def read_oulad(name, **kwargs):
    return pd.read_csv(os.path.join(RAW_DIR, name), na_values="?", **kwargs)

student_info = read_oulad("studentInfo.csv")
outcome_rank = {"Distinction": 0, "Pass": 1, "Fail": 2, "Withdrawn": 3}
info_dedup = (
    student_info.assign(_rank=student_info["final_result"].map(outcome_rank))
    .sort_values("_rank", ascending=False)
    .drop_duplicates(subset="id_student", keep="first")
)

SAMPLE_PER_OUTCOME = 8
rng = np.random.default_rng(42)
sample_ids, sample_labels = [], {}
for result in ["Distinction", "Pass", "Fail", "Withdrawn"]:
    ids = info_dedup.loc[info_dedup["final_result"] == result, "id_student"].values
    chosen = rng.choice(ids, size=min(SAMPLE_PER_OUTCOME, len(ids)), replace=False)
    sample_ids.extend(chosen.tolist())
    for sid in chosen:
        sample_labels[sid] = result

sample_ids = set(sample_ids)
print(f"Sampled {len(sample_ids)} real students across all 4 outcomes for inference")

Sampled 32 real students across all 4 outcomes for inference


In [7]:
vle_path = os.path.join(RAW_DIR, "studentVle.csv")
CHUNK = 1_000_000

weekly_clicks, weekly_days = {}, {}
for chunk in pd.read_csv(vle_path, na_values="?", chunksize=CHUNK,
                          usecols=["id_student", "date", "sum_click"]):
    chunk = chunk[chunk["id_student"].isin(sample_ids)]
    if chunk.empty:
        continue
    chunk["week"] = (chunk["date"].clip(lower=0) // 7).astype(int)
    chunk = chunk[chunk["week"] < MAX_WEEKS]

    g_clicks = chunk.groupby(["id_student", "week"])["sum_click"].sum()
    for (sid, wk), val in g_clicks.items():
        weekly_clicks[(sid, wk)] = weekly_clicks.get((sid, wk), 0) + val
    g_days = chunk.groupby(["id_student", "week"])["date"].apply(lambda s: len(set(s.unique())))
    for (sid, wk), n in g_days.items():
        weekly_days[(sid, wk)] = weekly_days.get((sid, wk), 0) + n

print(f"Weekly engagement collected for {len(set(sid for sid, wk in weekly_clicks))} of {len(sample_ids)} sampled students "
      f"({len(sample_ids) - len(set(sid for sid, wk in weekly_clicks))} had zero VLE activity)")

Weekly engagement collected for 32 of 32 sampled students (0 had zero VLE activity)


In [8]:
def build_sequence_tensor(student_ids, max_weeks, feature_cols, log1p_clicks):
    """Same shape as model_gru_oulad.ipynb's build_sequence_tensors(), for a
    small explicit list of students rather than the whole population."""
    X_list = []
    for sid in student_ids:
        seq = np.zeros((max_weeks, len(feature_cols)), dtype=np.float32)
        for wk in range(max_weeks):
            if (sid, wk) in weekly_clicks:
                seq[wk, 0] = weekly_clicks[(sid, wk)]
                seq[wk, 1] = weekly_days.get((sid, wk), 0)
        X_list.append(seq)
    X_tensor = torch.tensor(np.array(X_list), dtype=torch.float32)
    if log1p_clicks:
        X_tensor[:, :, 0] = torch.log1p(X_tensor[:, :, 0])
    return X_tensor

sample_id_list = sorted(sample_ids)
X_sample = build_sequence_tensor(sample_id_list, MAX_WEEKS, FEATURE_COLS, LOG1P_CLICKS)

with torch.no_grad():
    sample_proba = loaded_model(X_sample).numpy()

batch_results = pd.DataFrame({
    "id_student": sample_id_list,
    "actual_final_result": [sample_labels[sid] for sid in sample_id_list],
    "risk_score": sample_proba.round(4),
    "predicted_label": np.where(sample_proba >= RISK_THRESHOLD, "at_risk", "not_at_risk"),
})
batch_results.sort_values("risk_score", ascending=False)

,id_student,actual_final_result,risk_score,predicted_label
20,625214,Withdrawn,0.9993,at_risk
26,686034,Withdrawn,0.9992,at_risk
24,656637,Fail,0.9992,at_risk
30,2340190,Fail,0.9991,at_risk
9,555168,Withdrawn,0.9989,at_risk
3,134336,Fail,0.9989,at_risk
23,649469,Fail,0.9987,at_risk
1,100282,Withdrawn,0.9983,at_risk
5,406974,Withdrawn,0.9975,at_risk
19,618696,Fail,0.9934,at_risk


**Reading this:** `Withdrawn`/`Fail` students should generally show higher
`risk_score` than `Distinction`/`Pass` students, though with only 8 students
per outcome this is illustrative, not a real evaluation — `model_gru_oulad.ipynb`'s
full-population test-set metrics (96.6% precision / 82.4% recall) are the
trustworthy numbers, not this small sample.

## 5. Single-record inference — a hypothetical student's weekly clicks

In [9]:
# A student with strong early engagement that fades out by week 8 -- the kind
# of pattern the deep-dive EDA's engagement sequence heatmap flagged as a
# visual precursor to withdrawal.
weekly_click_pattern = {
    0: 45, 1: 52, 2: 48, 3: 40, 4: 35, 5: 20, 6: 15, 7: 8, 8: 3, 9: 0, 10: 0,
}
weekly_days_pattern = {
    0: 4, 1: 5, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 1, 8: 1, 9: 0, 10: 0,
}

seq = np.zeros((MAX_WEEKS, len(FEATURE_COLS)), dtype=np.float32)
for wk, clicks in weekly_click_pattern.items():
    seq[wk, 0] = clicks
for wk, days in weekly_days_pattern.items():
    seq[wk, 1] = days

X_new = torch.tensor(seq, dtype=torch.float32).unsqueeze(0)  # add batch dimension
if LOG1P_CLICKS:
    X_new[:, :, 0] = torch.log1p(X_new[:, :, 0])

with torch.no_grad():
    new_risk_score = loaded_model(X_new).item()

print(f"Risk score for this hypothetical fading-engagement student: {new_risk_score:.4f}")
print(f"Predicted label: {'at_risk' if new_risk_score >= RISK_THRESHOLD else 'not_at_risk'}")

Risk score for this hypothetical fading-engagement student: 0.9993
Predicted label: at_risk


## 6. Bonus — human-readable architecture/config summary (not the weights)

In [10]:
summary_path = os.path.join(MODELS_DIR, "risk_model_oulad_gru_v1_readable_summary.txt")
n_params = sum(p.numel() for p in loaded_model.parameters())

cfg = checkpoint_from_txt
lines = [
    "EngagementGRU -- architecture and configuration summary",
    "=" * 55,
    "",
    f"input_size:   {cfg['input_size']}",
    f"hidden_size:  {cfg['hidden_size']}",
    f"num_layers:   {cfg['num_layers']}",
    f"max_weeks:    {cfg['max_weeks']}",
    f"feature_cols: {cfg['feature_cols']}",
    f"log1p_clicks: {cfg['log1p_clicks']}",
    f"total trainable parameters: {n_params:,}",
    "",
    str(loaded_model),
]

with open(summary_path, "w") as f:
    f.write("\n".join(lines))

print(f"Saved: {summary_path}")
with open(summary_path) as f:
    print(f.read())

Saved: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_oulad_gru_v1_readable_summary.txt
EngagementGRU -- architecture and configuration summary

input_size:   2
hidden_size:  32
num_layers:   1
max_weeks:    40
feature_cols: ['total_clicks', 'days_active']
log1p_clicks: True
total trainable parameters: 4,001

EngagementGRU(
  (gru): GRU(2, 32, batch_first=True)
  (head): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


This file is genuinely human-readable — but it's documentation, **not a
reloadable model**. The actual weights only exist inside
`risk_model_oulad_gru_v1.txt` (Section 2) or the original `.pt` file, both
identical binary content.

## 7. Summary

- **Saved** `models/risk_model_oulad_gru_v1.txt` via `torch.save()` — byte-for-byte
  identical to the original `.pt` file, since PyTorch's save format doesn't
  depend on file extension at all (a stronger version of the same caveat as
  the XGBoost notebooks).
- **Verified round-trip:** a model reloaded *only* from the `.txt` file
  produces identical output to the original checkpoint on a fixed random input
  (max difference < 1e-6).
- **Batch inference** on a real sample of 32 students (8 per outcome),
  streamed fresh from `studentVle.csv` rather than reusing cached results.
- **Single-record inference** — a hand-built weekly click pattern (strong
  start, fading by week 8) scored end-to-end, showing how to call the model
  on genuinely new data.
- **Bonus:** a human-readable architecture/config summary — explicitly not the
  weights themselves, since (unlike XGBoost's trees) a GRU's raw parameters
  don't have a meaningful text form.